# Hyperparameter Tuning using Grid Search (MNIST)

### Additional Requirement
Install SciKeras before running this notebook:
pip install scikeras

In [ ]:
# ============================================================
# Hyperparameter Tuning using Grid Search (MNIST)
# ============================================================

import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import GridSearchCV
from scikeras.wrappers import KerasClassifier

# -----------------------------
# Load Dataset
# -----------------------------
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print("Training Images :", X_train.shape)
print("Testing Images  :", X_test.shape)

# -----------------------------
# Display Sample Images
# -----------------------------
plt.figure(figsize=(10,6))

for i in range(12):

    plt.subplot(3,4,i+1)

    plt.imshow(X_train[i], cmap="gray")

    plt.title(y_train[i])

    plt.axis("off")

plt.tight_layout()

plt.show()

# -----------------------------
# Normalize Images
# -----------------------------
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Flatten images
X_train = X_train.reshape(-1,784)
X_test = X_test.reshape(-1,784)

# -----------------------------
# Reduce Dataset Size
# (For Faster Grid Search)
# -----------------------------
X_train_small = X_train[:10000]
y_train_small = y_train[:10000]

print("Grid Search Dataset :", X_train_small.shape)

# -----------------------------
# Model Builder
# -----------------------------
def create_model(neurons=128, learning_rate=0.001):

    model = keras.Sequential([

        keras.layers.Input(shape=(784,)),

        keras.layers.Dense(
            neurons,
            activation="relu"
        ),

        keras.layers.Dense(
            64,
            activation="relu"
        ),

        keras.layers.Dense(
            10,
            activation="softmax"
        )

    ])

    optimizer = keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    model.compile(

        optimizer=optimizer,

        loss="sparse_categorical_crossentropy",

        metrics=["accuracy"]

    )

    return model

# -----------------------------
# Wrap Model
# -----------------------------
classifier = KerasClassifier(

    model=create_model,

    verbose=0

)

# -----------------------------
# Hyperparameter Grid
# -----------------------------
param_grid = {

    "model__neurons":[64,128,256],

    "model__learning_rate":[0.01,0.001],

    "batch_size":[32,64],

    "epochs":[5]

}

# -----------------------------
# Grid Search
# -----------------------------
grid = GridSearchCV(

    estimator=classifier,

    param_grid=param_grid,

    cv=3,

    scoring="accuracy",

    n_jobs=-1

)

grid_result = grid.fit(

    X_train_small,

    y_train_small

)

# -----------------------------
# Best Parameters
# -----------------------------
print("\nBest Accuracy :", grid_result.best_score_)

print("\nBest Parameters")

print(grid_result.best_params_)

# -----------------------------
# Train Best Model
# -----------------------------
best_model = create_model(

    neurons=grid_result.best_params_["model__neurons"],

    learning_rate=grid_result.best_params_["model__learning_rate"]

)

history = best_model.fit(

    X_train,

    y_train,

    epochs=5,

    batch_size=grid_result.best_params_["batch_size"],

    validation_split=0.2,

    verbose=1

)

# -----------------------------
# Evaluate
# -----------------------------
test_loss,test_accuracy = best_model.evaluate(

    X_test,

    y_test

)

print("\nTest Accuracy :",test_accuracy)

# -----------------------------
# Plot Accuracy
# -----------------------------
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)

plt.plot(

    history.history["accuracy"],

    label="Training"

)

plt.plot(

    history.history["val_accuracy"],

    label="Validation"

)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.title("Accuracy")

plt.legend()

# -----------------------------
# Plot Loss
# -----------------------------
plt.subplot(1,2,2)

plt.plot(

    history.history["loss"],

    label="Training"

)

plt.plot(

    history.history["val_loss"],

    label="Validation"

)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Loss")

plt.legend()

plt.tight_layout()

plt.show()

# -----------------------------
# Predictions
# -----------------------------
predictions = best_model.predict(X_test)

predicted_labels = np.argmax(predictions,axis=1)

# -----------------------------
# Display Predictions
# -----------------------------
plt.figure(figsize=(10,6))

for i in range(12):

    plt.subplot(3,4,i+1)

    plt.imshow(

        X_test[i].reshape(28,28),

        cmap="gray"

    )

    color="green" if predicted_labels[i]==y_test[i] else "red"

    plt.title(

        f"P:{predicted_labels[i]}\nT:{y_test[i]}",

        color=color

    )

    plt.axis("off")

plt.tight_layout()

plt.show()